In [ ]:
import numpy as np
import torch
import time
import math
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {DEVICE}")

# ==========================================
# 1. BUILD TARGET & F2 MATH
# ==========================================
def build_target_f2():
    T = torch.zeros((9, 9, 9), dtype=torch.int8, device=DEVICE)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1
    return T

T_target = build_target_f2()

def calc_f2_tensor(U, V, W):
    recon = torch.einsum("ir,jr,kr->ijk", U.float(), V.float(), W.float())
    return (recon.to(torch.int8) % 2)

# ==========================================
# 2. SIMULATED ANNEALING ENGINE
# ==========================================
def simulated_annealing_f2(U_full, V_full, W_full, col_drop, iterations=25000000):
    start_t = time.time()
    
    # Isolate 22 columns
    keep = [i for i in range(23) if i != col_drop]
    U = U_full[:, keep].clone()
    V = V_full[:, keep].clone()
    W = W_full[:, keep].clone()
    
    # Initial states
    current_recon = calc_f2_tensor(U, V, W)
    current_error = (current_recon != T_target).sum().item()
    
    best_error = current_error
    best_U, best_V, best_W = U.clone(), V.clone(), W.clone()
    
    # Annealing Schedule Parameters
    T_start = 2.0   # Lower start temp since we know the path to 26 is easy
    T_end = 0.005   # Deeper freeze to ruthlessly exploit the bottom of the valley
    cooling_factor = (T_end / T_start) ** (1.0 / iterations)
    
    temp = T_start
    
    print(f"Starting F2 Simulated Annealing | Dropping Col {col_drop}")
    print(f"Initial Errors: {current_error}/729")
    print(f"Schedule: {iterations} steps | Temp: {T_start} -> {T_end}\n")
    
    for i in range(iterations):
        # 1. Pick a random bit
        mat_choice = random.randint(0, 2)
        row = random.randint(0, 8)
        col = random.randint(0, 21)
        
        # 2. Flip the bit (0 -> 1 or 1 -> 0)
        if mat_choice == 0: U[row, col] = 1 - U[row, col]
        elif mat_choice == 1: V[row, col] = 1 - V[row, col]
        else: W[row, col] = 1 - W[row, col]
            
        # 3. Calculate new error
        new_recon = calc_f2_tensor(U, V, W)
        new_error = (new_recon != T_target).sum().item()
        
        # 4. Acceptance Logic (The Magic Step)
        delta_E = new_error - current_error
        accept = False
        
        if delta_E <= 0:
            accept = True # Always accept improvements or flat walks
        else:
            # Simulated Annealing formula: e^(-delta_E / Temp)
            probability = math.exp(-delta_E / temp)
            if random.random() < probability:
                accept = True # Accept a worse state to escape a crater!
                
        # 5. Apply or Revert
        if accept:
            current_error = new_error
            if current_error < best_error:
                best_error = current_error
                best_U, best_V, best_W = U.clone(), V.clone(), W.clone()
                print(f"  Step {i:6d} | Temp: {temp:.3f} | NEW BEST: {best_error}/729 errors")
                
                # --- NEW AUTOSAVE LOGIC ---
                # Saves instantly so you never lose the matrices to a disconnect
                np.savez('sa_checkpoint.npz', 
                         U=best_U.cpu().numpy(), 
                         V=best_V.cpu().numpy(), 
                         W=best_W.cpu().numpy())
                
            if best_error == 0:
                print(f"\n*** HISTORIC BREAKTHROUGH: EXACT RANK-22 F2 ROOT FOUND AT STEP {i}! ***")
                break
            
        # 6. Cool the temperature
        temp *= cooling_factor
        
        # Periodic heartbeat so you know it's alive during millions of steps
        if i > 0 and i % 1000000 == 0:
            print(f"  [Heartbeat] Step {i:8d} | Temp: {temp:.3f} | Current Best: {best_error}/729")
            
    print(f"\nDone in {time.time()-start_t:.1f}s | Final Best Error: {best_error}/729")
    return best_error, best_U, best_V, best_W

# ==========================================
# 3. EXECUTION BLOCK (Robust Loader)
# ==========================================
if __name__ == "__main__":
    print("Loading F2 Factorizations...")
    d = np.load('factorizations_f2.npz', allow_pickle=True)
    
    # EXPLICITLY load the 3x3x3 scheme
    scheme = d['3,3,3']
    
    # Safely extract if wrapped in a 0-d object array (The Fix)
    if isinstance(scheme, np.ndarray) and scheme.ndim == 0:
        scheme = scheme.item()
        
    # Route based on dict or pure array
    if isinstance(scheme, dict):
        U_f2 = torch.tensor(scheme['u'], dtype=torch.int8, device=DEVICE)
        V_f2 = torch.tensor(scheme['v'], dtype=torch.int8, device=DEVICE)
        W_f2 = torch.tensor(scheme['w'], dtype=torch.int8, device=DEVICE)
    else:
        U_f2 = torch.tensor(scheme[0], dtype=torch.int8, device=DEVICE)
        V_f2 = torch.tensor(scheme[1], dtype=torch.int8, device=DEVICE)
        W_f2 = torch.tensor(scheme[2], dtype=torch.int8, device=DEVICE)
        
    print(f"Successfully loaded 3x3x3 F2 scheme: {U_f2.shape}\n")
    
    # Let the annealing engine run!
    simulated_annealing_f2(U_f2, V_f2, W_f2, col_drop=0, iterations=25000000)

Executing on: cuda
Loading F2 Factorizations...
Successfully loaded 3x3x3 F2 scheme: torch.Size([9, 23])

Starting F2 Simulated Annealing | Dropping Col 0
Initial Errors: 46/729
Schedule: 25000000 steps | Temp: 2.0 -> 0.005

  Step   2769 | Temp: 1.999 | NEW BEST: 45/729 errors
  Step   3094 | Temp: 1.999 | NEW BEST: 42/729 errors
  Step   3110 | Temp: 1.999 | NEW BEST: 41/729 errors
  Step   4940 | Temp: 1.998 | NEW BEST: 36/729 errors
  Step   5000 | Temp: 1.998 | NEW BEST: 33/729 errors
  Step   5046 | Temp: 1.998 | NEW BEST: 30/729 errors
  Step   5202 | Temp: 1.998 | NEW BEST: 27/729 errors
  Step 133181 | Temp: 1.937 | NEW BEST: 26/729 errors
  [Heartbeat] Step  1000000 | Temp: 1.574 | Current Best: 26/729
  [Heartbeat] Step  2000000 | Temp: 1.238 | Current Best: 26/729
  [Heartbeat] Step  3000000 | Temp: 0.975 | Current Best: 26/729
  [Heartbeat] Step  4000000 | Temp: 0.767 | Current Best: 26/729
  [Heartbeat] Step  5000000 | Temp: 0.603 | Current Best: 26/729
  [Heartbeat] Step